In [ ]:
import glob
import os

import numpy as np
import rasterio
from matplotlib import pyplot as plt

# Visualize gap-filling evaluation samples from the folder structure.
#
# Layout: one row per sample, 7 columns:
#   t0 | t1 | t2 (gappy) | t3 | t4 | GT clean | gap mask
#
# Data layout expected:
#   <gap_fill_root>/SSL-consecutive/{idx:06d}/{timestamp}/S2C.tif  (10-band: B2..B12)
#   <gap_fill_root>/ground_truth/{idx:06d}/{t2_timestamp}/S2C.tif  gap_mask.tif

_s2_data_dir = os.environ.get(
    "S2_DATA_DIR",
    "../../../sentinel/data",
)
gap_fill_root = os.path.join(_s2_data_dir, "gap_filling")
ssl_dir = os.path.join(gap_fill_root, "SSL-consecutive")
gt_dir = os.path.join(gap_fill_root, "ground_truth")

n_samples = 5
n_timesteps = 5
seed = None  # set to None for non-deterministic sampling
MIDDLE_T = n_timesteps // 2  # index 2


def robust_rgb(
    b4: np.ndarray, b3: np.ndarray, b2: np.ndarray,
    pmin: float = 2, pmax: float = 98,
    lo: float = None, hi: float = None,
):
    rgb = np.stack([b4, b3, b2], axis=-1).astype(np.float32)
    gaps = np.any(np.isnan(rgb), axis=-1)
    if lo is None or hi is None:
        lo = np.nanpercentile(rgb, pmin)
        hi = np.nanpercentile(rgb, pmax)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = 0.0, 2500.0
    rgb = (rgb - lo) / (hi - lo + 1e-6)
    rgb = np.clip(rgb, 0, 1)
    rgb[gaps] = 0.0
    return rgb, gaps


def read_raw_rgb(timestep_dir: str):
    """Return (b4, b3, b2) raw float32 arrays from S2C.tif without normalization."""
    try:
        with rasterio.open(os.path.join(timestep_dir, "S2C.tif")) as src:
            b2 = src.read(1).astype(np.float32)
            b3 = src.read(2).astype(np.float32)
            b4 = src.read(3).astype(np.float32)
        return b4, b3, b2
    except Exception:
        dummy = np.full((64, 64), np.nan)
        return dummy, dummy.copy(), dummy.copy()


def read_rgb(timestep_dir: str, lo: float = None, hi: float = None):
    """Return (rgb (H,W,3), gaps (H,W bool), height, width)."""
    try:
        b4, b3, b2 = read_raw_rgb(timestep_dir)
        rgb, gaps = robust_rgb(b4, b3, b2, lo=lo, hi=hi)
        return rgb, gaps, b4.shape[0], b4.shape[1]
    except Exception:
        dummy = np.zeros((64, 64, 3), dtype=np.float32)
        return dummy, np.zeros((64, 64), dtype=bool), None, None


def read_gap_mask(gt_patch_dir: str) -> np.ndarray:
    """Return uint8 (H, W) binary mask from gap_mask.tif."""
    try:
        with rasterio.open(os.path.join(gt_patch_dir, "gap_mask.tif")) as src:
            return src.read(1).astype(np.float32)
    except Exception:
        return np.zeros((64, 64), dtype=np.float32)


def date_label(folder_name: str) -> str:
    d = folder_name[:8]
    try:
        return f"{d[:4]}/{d[4:6]}/{d[6:8]}"
    except Exception:
        return d


# --- collect sample indices that have both SSL-consecutive and ground_truth entries ---

rng = np.random.default_rng(seed) if seed is not None else np.random.default_rng()

chosen = []
seen = 0
try:
    with os.scandir(ssl_dir) as it:
        for entry in it:
            if not entry.is_dir():
                continue
            if not os.path.isdir(os.path.join(gt_dir, entry.name)):
                continue  # skip samples without a matching ground truth
            seen += 1
            if len(chosen) < n_samples:
                chosen.append(entry.name)
            else:
                j = int(rng.integers(0, seen))
                if j < n_samples:
                    chosen[j] = entry.name
except FileNotFoundError:
    pass

if not chosen:
    raise FileNotFoundError(
        f"No gap-filling samples found.\n"
        f"Expected SSL-consecutive at: {ssl_dir}\n"
        f"Expected ground_truth at:    {gt_dir}"
    )

chosen = sorted(chosen)
n_found = len(chosen)

# --- plot ---
# 7 columns: t0 t1 t2-gappy t3 t4 | GT-clean | gap-mask

N_COLS = n_timesteps + 2
fig, axes = plt.subplots(n_found, N_COLS, figsize=(3.5 * N_COLS, 4.2 * n_found))
if n_found == 1:
    axes = [axes]

for row, idx_name in enumerate(chosen):
    ssl_sample = os.path.join(ssl_dir, idx_name)
    gt_sample = os.path.join(gt_dir, idx_name)

    ts_dirs = sorted(
        d for d in glob.glob(os.path.join(ssl_sample, "*")) if os.path.isdir(d)
    )
    gt_ts_dirs = sorted(
        d for d in glob.glob(os.path.join(gt_sample, "*")) if os.path.isdir(d)
    )

    # -- compute shared lo/hi from all TIF panels in this row --
    all_raw = []
    for tdir in ts_dirs + gt_ts_dirs:
        for band_arr in read_raw_rgb(tdir):
            valid = band_arr[~np.isnan(band_arr)]
            if valid.size:
                all_raw.append(valid)
    if all_raw:
        combined = np.concatenate(all_raw)
        row_lo = float(np.percentile(combined, 2))
        row_hi = float(np.percentile(combined, 98))
    else:
        row_lo, row_hi = 0.0, 2500.0
    if not np.isfinite(row_lo) or not np.isfinite(row_hi) or row_hi <= row_lo:
        row_lo, row_hi = 0.0, 2500.0

    # -- time series columns (t0 .. t4) --
    for t in range(n_timesteps):
        ax = axes[row][t]

        if t >= len(ts_dirs):
            ax.set_visible(False)
            continue

        rgb, gaps, h, w = read_rgb(ts_dirs[t], lo=row_lo, hi=row_hi)
        sz = f"  {h}×{w}" if h is not None else ""
        frac = 100.0 * float(gaps.sum()) / max(gaps.size, 1)

        if t == MIDDLE_T:
            title = (
                f"t={t} (gappy)\n"
                f"{date_label(os.path.basename(ts_dirs[t]))}{sz}\n"
                f"synthetic gaps={frac:.1f}%"
            )
        else:
            title = f"t={t}\n{date_label(os.path.basename(ts_dirs[t]))}{sz}\ngaps={frac:.1f}%"

        ax.imshow(rgb)
        ax.set_title(title, fontsize=7)
        ax.axis("off")

    # -- ground truth column --
    ax_gt = axes[row][n_timesteps]
    if gt_ts_dirs:
        gt_rgb, _, h, w = read_rgb(gt_ts_dirs[0], lo=row_lo, hi=row_hi)
        sz = f"  {h}×{w}" if h is not None else ""
        ax_gt.imshow(gt_rgb)
        ax_gt.set_title(
            f"Ground truth (t=2)\n"
            f"{date_label(os.path.basename(gt_ts_dirs[0]))}{sz}\n"
            f"(original clean)",
            fontsize=7,
        )
    else:
        ax_gt.text(0.5, 0.5, "GT missing", ha="center", va="center", transform=ax_gt.transAxes)
    ax_gt.axis("off")

    # highlight the ground truth column with a subtle border
    for spine in ax_gt.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#1a7f1a")
        spine.set_linewidth(2)

    # -- gap mask column --
    ax_mask = axes[row][n_timesteps + 1]
    if gt_ts_dirs:
        mask = read_gap_mask(gt_ts_dirs[0])
        coverage = 100.0 * float(mask.sum()) / max(mask.size, 1)
        ax_mask.imshow(mask, cmap="Reds", vmin=0, vmax=1)
        ax_mask.set_title(
            f"Gap mask\ncoverage={coverage:.1f}%\n(red=gap)",
            fontsize=7,
        )
    else:
        ax_mask.text(0.5, 0.5, "mask missing", ha="center", va="center", transform=ax_mask.transAxes)
    ax_mask.axis("off")

    # left margin: sample id
    axes[row][0].set_ylabel(f"sample\n{idx_name}", fontsize=8, labelpad=4)
    axes[row][0].yaxis.set_visible(True)

fig.suptitle(
    f"Gap-filling evaluation dataset  ({n_found} samples)\n"
    "Middle frame (t=2) has synthetic NaN gaps (black). "
    "Green-bordered column is the original clean ground truth.",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()
plt.show()